# AWEL Research

In [6]:
from dbgpt.core.awel import DAG, MapOperator

In [10]:
with DAG("awel_hello_world") as dag:
    task = MapOperator(map_function=lambda x: print(f"Hello, {x}!"))
output = task.call(call_data="world")

In [17]:
print(output.cr_code)

<code object call at 0x0000025A75B64190, file "d:\developer\projects\jupyterprojects\db-gpt\db-gpt\dbgpt\core\awel\operators\base.py", line 187>


In [18]:
import asyncio

In [21]:
from dbgpt.core.awel import DAG, MapOperator, TriggerOperator, SimpleCallDataInputSource

In [55]:
with DAG("awel_hello_world") as dag:
    input_task = TriggerOperator()
    task = MapOperator(map_function=lambda x: print(f"Hello, {x}!"))
    input_task >> task

task.call(call_data="world")

<coroutine object BaseOperator.call at 0x0000025A73E9FB50>

In [56]:
await task.call(call_data="world")

Hello, world!


## an easy example

### Map Operator

In [43]:
import asyncio
from dbgpt.core.awel import DAG, MapOperator

class DoubleNumberOperator(MapOperator[int, int]):
    async def map(self, x: int) -> int:
        print(f"Received {x}, returning {x * 2}")
        return x * 2 
with DAG("awel_double_number") as dag:
    task = DoubleNumberOperator() 

In [52]:
task_test = task.call(2)

In [53]:
assert await task_test

Received 2, returning 4


### Reduce Operator

In [1]:
import asyncio
from typing import AsyncIterator
from dbgpt.core.awel import DAG, ReduceStreamOperator, StreamifyAbsOperator

In [59]:
class NumberProducerOperator(StreamifyAbsOperator[int, int]):
    async def streamify(self, n: int) -> AsyncIterator[int]:
        for i in range(n):
            yield i

class MySumOperator(ReduceStreamOperator[int,int]):
    async def reduce(self, x:int, y: int) -> int:
        return x + y

with DAG("sum_numbers_dag") as dag:
    task = NumberProducerOperator()
    sum_task = MySumOperator()
    task >> sum_task

In [63]:
sum_task_in_case = sum_task.call(call_data=10)

In [64]:
await sum_task_in_case

45

### Join Operator

In [1]:
import asyncio

In [2]:
from dbgpt.core.awel import (
    DAG, JoinOperator, MapOperator, InputOperator, SimpleCallDataInputSource
)

In [3]:
SimpleCallDataInputSource()

In [6]:
with DAG("sum_numbers_dag") as dag:
    input_task = InputOperator(input_source=SimpleCallDataInputSource())
    task1 = MapOperator(map_function=lambda x: x["t1"])
    task2 = MapOperator(map_function=lambda x: x["t2"])
    sum_task = JoinOperator(combine_function=lambda x, y: x + y)
    input_task >> task1 >> sum_task
    input_task >> task2 >> sum_task

In [9]:
await sum_task.call(call_data={"t1": 5, "t2": 8})

13

In [10]:
if await sum_task.call(call_data={"t1": 5, "t2": 8}) == 13:
    print("Success!")
else:
    print("Failed")

Success!


### Branch Operator

In [2]:
import asyncio
from dbgpt.core.awel import (
    DAG, BranchOperator, MapOperator, JoinOperator,
    InputOperator, SimpleCallDataInputSource,
    is_empty_data
)

In [6]:
def branch_even(x: int) -> bool:
    return x % 2 == 0

def branch_odd(x: int) -> bool:
    return not branch_even(x)

branch_mapping = {
    branch_even: "even_task",
    branch_odd: "odd_task"
}

def even_func(x: int) -> int:
    print(f"Branch even, {x} is even, multiply by 10")
    return x * 10

def odd_func(x: int) -> int:
    print(f"Branch odd, {x} is odd, multiply by itself")
    return x * x

def combine_function(x: int, y: int) -> int:
    print(f"Received {x} and {y}")
    return x if not is_empty_data(x) else y

with DAG("awel_branch_operator") as dag:
    input_task = InputOperator(input_source=SimpleCallDataInputSource())
    task = BranchOperator(branches=branch_mapping)
    even_task = MapOperator(task_name="even_task", map_function=even_func)
    odd_task = MapOperator(task_name="odd_task", map_function=odd_func)
    join_task = JoinOperator(combine_function=combine_function)
    
    input_task >> task >> even_task >> join_task
    input_task >> task >> odd_task >> join_task

print("First call, input is 5")
await join_task.call(call_data=5) == 25
print("=" * 80)
#print("Second call, input is 6")
#await join_task.call(call_data=6) == 60

First call, input is 5
Branch odd, 5 is odd, multiply by itself
Received EmptyData(SKIP_DATA) and 25


In [12]:
import contextlib
import  sys

In [7]:
# 返回上下文管理器
def _temporary_variable(name, value):
    try:
        setattr(sys.modules[__name__], name, value)
        return contextlib.suppress(AttributeError)  # 返回上下文管理器
    except AttributeError:
        return contextlib.ExitStack().enter_context(contextlib.suppress(AttributeError))  # 返回上下文管理器

In [8]:
#对上述branch函数进行对接
branch_functions = {
    branch_mapping[branch_even]: even_func,
    branch_mapping[branch_odd]: odd_func
}

In [10]:
# 查看branch分支中，对接后的字典样式
print(branch_functions)

{'even_task': <function even_func at 0x00000162B894A0E0>, 'odd_task': <function odd_func at 0x00000162B894A7A0>}


In [20]:
#对上述dag语句进行模板功能固化
def branch_using_dag(input_functions_dict: dict):
    task_function_list = []
    with DAG("awel_branch_operator") as dag:
        input_task = InputOperator(input_source=SimpleCallDataInputSource())
        task = BranchOperator(branches=branch_mapping)
        join_task = JoinOperator(combine_function=combine_function)
        with contextlib.ExitStack() as stack:
            for func, executor_function in input_functions_dict.items():
                task_name = func
                stack.enter_context(_temporary_variable(func, executor_function))
                func = MapOperator(task_name=task_name, map_function=executor_function)
                input_task >> task >> func >> join_task
        return join_task

In [21]:
#套用模板后，测试调用情况
branch_task = branch_using_dag(input_functions_dict=branch_functions)
print("First call, input is 5")
await branch_task.call(call_data=5)
print("=" * 80)

First call, input is 5
Branch odd, 5 is odd, multiply by itself
Received EmptyData(SKIP_DATA) and 25


成功执行，表明结合node的function特性，进行模板功能固化的方式是可行的，这样不论branch分支中涉及多个判断路径和函数，都可以套用上述模板进行自动遍历调用，而无需重复进行task传入传出

再重新思考一个问题，上述对接函数是否可以封装入BranchOperator中呢？考虑到在BranchOperator中，是包含所谓的决策逻辑的，但决策后则需要通过字典或者其他方式进行后续执行function的指定，才能跟MapOperator进行对接，否则就只能实现不同路径的分支判断而已。

较优的思路是，在BranchOperator中不仅能够进行多情况判断，而且能够保证在判断后，可以进一步执行相关function

In [ ]:
# 通过

### Http Trigger

In [1]:
from dbgpt.core.awel import DAG, HttpTrigger, MapOperator, setup_dev_environment

with DAG("awel_hello_world") as dag:
    trigger_task = HttpTrigger(endpoint="/awel_tutorial/hello_world")
    task = MapOperator(map_function=lambda x: f"Hello, world!")
    trigger_task >> task  
    
setup_dev_environment([dag], port=3333)

2024-03-21 21:27:13 DESKTOP-J06KNTU dbgpt.component[6024] INFO Register component with name dbgpt_awel_trigger_manager and instance: <dbgpt.core.awel.trigger.trigger_manager.DefaultTriggerManager object at 0x00000251B42A3820>
2024-03-21 21:27:14 DESKTOP-J06KNTU dbgpt.core.awel.dag.base[6024] INFO Writing Mermaid syntax to dag-vis-awel_hello_world.md
2024-03-21 21:27:14 DESKTOP-J06KNTU dbgpt.core.awel[6024] WARNING Visualize DAG DAG(dag_id=awel_hello_world) failed: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH, if your system has no graphviz, you can install it by `pip install graphviz` or `sudo apt install graphviz`
2024-03-21 21:27:14 DESKTOP-J06KNTU dbgpt.core.awel.trigger.trigger_manager[6024] INFO Register trigger HttpTrigger(node_id=f2e586f6-63b9-4c1f-8f21-33b37f76d7cf)
2024-03-21 21:27:14 DESKTOP-J06KNTU dbgpt.core.awel.trigger.http_trigger[6024] INFO mount router function <function HttpTrigger._create_route_func.<locals>.creat

HttpTrigger(node_id=f2e586f6-63b9-4c1f-8f21-33b37f76d7cf)
 -> MapOperator(node_id=2575c5ce-7ba2-48b9-abc4-7bb1ed39f9ea)
False False 1
True


D:\Developer\Application_Developer\Conda\envs\dbgpt_env\lib\site-packages\uvicorn\main.py:579: RuntimeWarning: coroutine 'Server.run_in_jupyter' was never awaited
  server.run_in_jupyter() #run single server function in jupyter


SystemExit: 3

D:\Developer\Application_Developer\Conda\envs\dbgpt_env\lib\site-packages\IPython\core\interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
%tb

SystemExit: 3

## RAG With AWEL

In [1]:
### Prepare Embedding Model
from dbgpt.rag.embedding import DefaultEmbeddingFactory
embeddings = DefaultEmbeddingFactory.default(r"D:\Developer\Projects\pythonProject\reference_project\DB-GPT\DB-GPT\models\m3e-large")

Current platform is windows, use avx2 as default cpu architecture


D:\Developer\Application_Developer\Conda\envs\dbgpt_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No sentence-transformers model found with name D:\Developer\Projects\pythonProject\reference_project\DB-GPT\DB-GPT\models\m3e-large. Creating a new one with MEAN pooling.
D:\Developer\Application_Developer\Conda\envs\dbgpt_env\lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [3]:
### Load Knowledge and store in vector store
import asyncio
import shutil
from dbgpt.core.awel import DAG
from dbgpt.rag import ChunkParameters
from dbgpt.rag.knowledge import KnowledgeType
from dbgpt.rag.operators import EmbeddingAssemblerOperator, KnowledgeOperator
from dbgpt.storage.vector_store.chroma_store import ChromaVectorConfig
from dbgpt.storage.vector_store.connector import VectorStoreConnector


In [4]:
# Delete old vector store directory(/tmp/awel_rag_test_vector_store)
shutil.rmtree("/tmp/awel_rag_test_vector_store", ignore_errors=True)

In [30]:
vector_connector = VectorStoreConnector.from_default(
    "Chroma",
    vector_store_config = ChromaVectorConfig(
        name="test_vstore",
        persist_path="/tmp/awel_rag_test_vector_store",
    ),
    embedding_fn=embeddings
)

with DAG("load_knowledge_dag") as knowledge_dag:
    # Load knowledge from URL
    knowledge_task = KnowledgeOperator(knowledge_type=KnowledgeType.URL.name)
    assembler_task = EmbeddingAssemblerOperator(
        vector_store_connector=vector_connector,
        chunk_parameters=ChunkParameters(chunk_strategy="CHUNK_BY_SIZE")
    )
    knowledge_task >> assembler_task

chunks = await assembler_task.call("https://docs.dbgpt.site/docs/latest/awel/")
print(f"Chunk length: {len(chunks)}")

<class 'dbgpt.storage.vector_store.chroma_store.ChromaStore'>
Chunk length: 14


In [6]:
# Retrieve Knowledge From Vector Store
from dbgpt.core.awel import MapOperator
from dbgpt.rag.operators import EmbeddingRetrieverOperator

with DAG("retriever_dag") as retriever_dag:
    retriever_task = EmbeddingRetrieverOperator(
        top_k=3,
        vector_store_connector=vector_connector,
    )
    content_task = MapOperator(lambda cks: "\n".join(c.content for c in cks))
    retriever_task >> content_task

chunks = await content_task.call("awel is what?")
print(chunks)

Agentic Workflow Expression Language(AWEL) is a set of intelligent agent workflow expression language specially designed for large model application
development. It provides great functionality and flexibility. Through the AWEL API, you can focus on the development of business logic for LLMs applications
without paying attention to cumbersome model and environment details.
AWEL adopts a layered API design. AWEL's layered API design architecture is shown in the figure below.

AWEL Design​
AWEL(Agentic Workflow Expression Language) | DB-GPT









Skip to main content⭐️ If you like DB-GPT, give it a star on GitHub! ⭐️Docs中文文档Communityv0.5.2v0.5.2v0.5.1v0.5.0HuggingFaceOverviewQuickstartapiapplicationapplication_manualWhat is AWEL?awel_tutorialcookbookGet StartedWhy use AWEL?AWEL(Agentic Workflow Expression Language)changelogcookbookfaqFAQinstallationmodulesupgradeUse CasesWhat is AWEL?Version: v0.5.2On this pageAWEL(Agentic Workflow Expression Language)
AWEL Design​
AWEL is divided int

In [14]:
# what is the retriever_dag
print(type(retriever_dag))

<class 'dbgpt.core.awel.dag.base.DAG'>


In [7]:
## input api key
import os
import getpass

os.environ['CHATGLM4_API_KEY'] = getpass.getpass('CHATGLM4_API_KEY:')

CHATGLM4_API_KEY: ········


In [8]:
input_api_key = os.environ['CHATGLM4_API_KEY']

In [25]:
## laod LLM model
from dbgpt.model.proxy import ZhipuLLMClient
llm_client = ZhipuLLMClient(api_key=input_api_key)
#llm_client = ZhipuLLMClient(api_base="https://open.bigmodel.cn/api/paas/v4", api_key=input_api_key)

In [11]:
# create rag program
from dbgpt.core.awel import InputOperator, JoinOperator, InputSource
from dbgpt.core.operators import PromptBuilderOperator, RequestBuilderOperator
from dbgpt.model.operators import LLMOperator

In [12]:
prompt = """Based on the known information below, provide users with professional and concise answers to their questions. 
If the answer cannot be obtained from the provided content, please say: 
"The information provided in the knowledge base is not sufficient to answer this question.". 
It is forbidden to make up information randomly. When answering, it is best to summarize according to points 1.2.3.
          known information: 
          {context}
          question:
          {question}
"""

In [31]:
with DAG("ll,_rag_dag") as rag_dag:
    input_task = InputOperator(input_source=InputSource.from_callable())
    #retriever_dag
    retriever_task = EmbeddingRetrieverOperator(
        top_k=3,
        vector_store_connector=vector_connector
    )
    content_task = MapOperator(lambda cks: "\n".join(c.content for c in cks))
    merge_task = JoinOperator(lambda context, question: {"context": context, "question": question})
    prompt_task = PromptBuilderOperator(prompt)
    # The model is chatglm4, you can replace it with other models
    req_build_task = RequestBuilderOperator(model="glm-4")
    llm_task = LLMOperator(llm_client=llm_client)
    result_task = MapOperator(lambda r: r.text)
    input_task >> retriever_task >> content_task >> merge_task
    input_task >> merge_task

    merge_task >> prompt_task >> req_build_task >>llm_task >> result_task

print(await merge_task.call("What is the awel?"))

{'context': "Agentic Workflow Expression Language(AWEL) is a set of intelligent agent workflow expression language specially designed for large model application\ndevelopment. It provides great functionality and flexibility. Through the AWEL API, you can focus on the development of business logic for LLMs applications\nwithout paying attention to cumbersome model and environment details.\nAWEL adopts a layered API design. AWEL's layered API design architecture is shown in the figure below.\n\nAWEL Design\u200b\nAWEL(Agentic Workflow Expression Language) | DB-GPT\n\n\n\n\n\n\n\n\n\nSkip to main content⭐️ If you like DB-GPT, give it a star on GitHub! ⭐️Docs中文文档Communityv0.5.3v0.5.3v0.5.2v0.5.1v0.5.0HuggingFaceOverviewQuickstartapiapplicationapplication_manualWhat is AWEL?awel_tutorialcookbookGet StartedWhy use AWEL?AWEL(Agentic Workflow Expression Language)changelogcookbookfaqFAQinstallationmodulesupgradeUse CasesWhat is AWEL?Version: v0.5.3On this pageAWEL(Agentic Workflow Expression La

In [27]:
rag_dag.print_tree()

InputOperator(node_id=f1d3e2e2-9239-476e-b46d-40c2f551521e)
 -> EmbeddingRetrieverOperator(node_id=e1871f05-8dab-40ed-a2ba-9ec188937f67)
|  -> MapOperator(node_id=acf0b80f-e544-4385-9911-cc75572aaca5)
|    -> JoinOperator(node_id=1fc8d246-655f-43b2-bfc2-ff7305138dce)
|      -> PromptBuilderOperator(node_id=f426a51a-e6dd-41be-828a-f29789f38147)
|        -> RequestBuilderOperator(node_id=7fd05864-54a4-4974-80c9-3ccea8c25611)
|          -> LLMOperator(node_id=cc330165-7118-4c48-bbe2-44a155a49243)
|            -> MapOperator(node_id=e0f391c2-d20b-4a8d-a574-f78bcc45a7a0)
 -> JoinOperator(node_id=1fc8d246-655f-43b2-bfc2-ff7305138dce)
   -> PromptBuilderOperator(node_id=f426a51a-e6dd-41be-828a-f29789f38147)
     -> RequestBuilderOperator(node_id=7fd05864-54a4-4974-80c9-3ccea8c25611)
       -> LLMOperator(node_id=cc330165-7118-4c48-bbe2-44a155a49243)
         -> MapOperator(node_id=e0f391c2-d20b-4a8d-a574-f78bcc45a7a0)


## Mutl-source RAG

### Part1: Knowledge Construction

In [1]:
# chunk management
# find where is it: DB-GPT\dbgpt\rag\chunk_manager.py
# let's try it
from dbgpt.rag.chunk_manager import ChunkManager, ChunkParameters
from dbgpt.rag.knowledge.factory import KnowledgeFactory
from dbgpt.rag.knowledge.base import KnowledgeType, ChunkStrategy

In [2]:
# get a knowledge object
knowledge_factory = KnowledgeFactory()
# import the pdf.file as document knowledge
document_path = r"D:\Research\AGI\2312.17449_db_gpt_paper.pdf"
document_knowledge = knowledge_factory.create(datasource=document_path, knowledge_type=KnowledgeType.DOCUMENT)

In [3]:
print(type(document_knowledge))

<class 'dbgpt.rag.knowledge.pdf.PDFKnowledge'>


In [4]:
# load pdf content
document_content = document_knowledge.load()

In [5]:
paper_chunk_parameters = ChunkParameters(chunk_strategy="CHUNK_BY_PARAGRAPH")

In [6]:
chunk_manager = ChunkManager(document_knowledge)

In [7]:
knowledge_split = chunk_manager.split(document_content)

In [8]:
print(type(knowledge_split))

<class 'list'>


In [9]:
print(len(knowledge_split))

108


In [10]:
print(knowledge_split[0:2])

[Chunk(content='DB-GPT: Empowering Database Interactions with\nPrivate Large Language Models\nSiqiao Xue♢, Caigao Jiang♢, Wenhui Shi♢, Fangyin Cheng♥, Keting Chen♢, Hongjun Yang♢,\nZhiping Zhang♡, Jianshan He♢, Hongyang Zhang♦, Ganglin Wei♢, Wang Zhao,\nFan Zhou♢, Danrui Qi♣, Hong Yi, Shaodong Liu♠, Faqiang Chen♢,∗\n♢Ant Group,♡Alibaba Group,♥JD Group,♠Meituan\n♦Southwestern University of Finance and Economics, China\n♣Simon Fraser University, Canada\n{siqiao.xsq,caigao.jcg,faqiang.cfq}@antgroup.com\nAbstract', metadata={'source': 'D:\\Research\\AGI\\2312.17449_db_gpt_paper.pdf', 'page': 0}, chunk_id='0ac67f66-d512-4113-8acc-249d8a17a259', score=0.0, summary='', separator='\n'), Chunk(content='Abstract\nThe recent breakthroughs in large language models (LLMs) are positioned to\ntransition many areas of software. Database technologies particularly have an\nimportant entanglement with LLMs as efficient and intuitive database interactions\nare paramount. In this paper, we present DB-GPT, 

In [11]:
# Now we get chunks, we can do the inverted Indexing
from dbgpt.storage.metadata.db_factory import UnifiedDBManagerFactory
from dbgpt.storage.metadata.db_manager import DatabaseManager

In [12]:
from dbgpt.component import SystemApp
db_system_app = SystemApp()
db_manager = DatabaseManager()

In [13]:
url = f"sqlite:///:knowledge_limit:"
engine_args = {
    "pool_size": 10,
    "max_overflow": 20,
    "pool_timeout": 30,
    "pool_recycle": 3600,
    "pool_pre_ping": True,
}
db_manager.init_db(url, engine_args=engine_args)

In [14]:
db_factory = UnifiedDBManagerFactory(system_app=db_system_app, db_manager=db_manager)

In [15]:
db_manager = db_factory.create()

In [16]:
# Now we embedding this chunk, and go on see what will happen
from dbgpt.rag.embedding.embedding_factory import DefaultEmbeddingFactory
embedding_model = DefaultEmbeddingFactory.call_embeddings_model_from_huggingface("BAAI/bge-base-en")

D:\Developer\Application_Developer\Conda\envs\dbgpt_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
# now we get an embedding model in our projects'default
# we want to embedding chunks first
from dbgpt.rag.operators import EmbeddingAssemblerOperator, KnowledgeOperator
from dbgpt.storage.vector_store.chroma_store import ChromaVectorConfig
from dbgpt.storage.vector_store.connector import VectorStoreConnector

In [ ]:
shutil.rmtree("/tmp/awel_rag_test_vector_store", ignore_errors=True)

In [18]:
knowledge_vector_connector = VectorStoreConnector.from_default(
    "Chroma",
    vector_store_config = ChromaVectorConfig(
        name="test_vstore",
        persist_path="/tmp/awel_rag_test_vector_store",
    ),
    embedding_fn=embedding_model
)

<class 'dbgpt.storage.vector_store.chroma_store.ChromaStore'>


In [19]:
knowledge_embedding_assembler = EmbeddingAssemblerOperator(
    vector_store_connector=knowledge_vector_connector,
    chunk_parameters=paper_chunk_parameters
)

In [20]:
embedding_chunks = knowledge_embedding_assembler.assemble(document_knowledge)
print(len(embedding_chunks))

666


In [21]:
# We can see what is embedding_chunks
print(embedding_chunks[0])

content='DB-GPT: Empowering Database Interactions with' metadata={'source': 'D:\\Research\\AGI\\2312.17449_db_gpt_paper.pdf', 'page': 0} chunk_id='2aeb3c6f-762a-408e-a0e3-296859dff691' score=0.0 summary='' separator='\n'


In [22]:
print(embedding_chunks[1])

content='Private Large Language Models' metadata={'source': 'D:\\Research\\AGI\\2312.17449_db_gpt_paper.pdf', 'page': 0} chunk_id='531252eb-293e-4d18-9762-b8e12777593b' score=0.0 summary='' separator='\n'


### Part2:Knowledge Retriever

In [23]:
# Now we can retrieve Query relative info from this chunks
from dbgpt.rag.operators.embedding import EmbeddingRetrieverOperator

In [24]:
# we do an easy retrieval
embedding_retriever_case = EmbeddingRetrieverOperator(
    vector_store_connector=knowledge_vector_connector,
    top_k = 10
)

In [25]:
retriever_chunks = embedding_retriever_case.retrieve("what is smmf")

In [26]:
print(
    f"embedding retriever results:{[chunk.content for chunk in retriever_chunks]}"
)

embedding retriever results:['4.3 SMMF Evaluation', 'B.3 SMMF Evaluation Details']


In [27]:
# We want to rerank the chunks, and rewrite query
from dbgpt.rag.retriever.rerank import Ranker,RRFRanker,DefaultRanker
from dbgpt.rag.retriever.rewrite import QueryRewrite

In [28]:
# if we input rerank, and we can see what's content is different
embedding_retriever_sec_case = EmbeddingRetrieverOperator(
    vector_store_connector=knowledge_vector_connector,
    top_k = 10,
    rerank=DefaultRanker(
        topk=3
    )
)

In [29]:
sec_retriever_chunks = embedding_retriever_sec_case.retrieve("what is SMMF>")

In [30]:
print(
    f"embedding retriever results:{[chunk.content for chunk in sec_retriever_chunks]}"
)

embedding retriever results:['4.3 SMMF Evaluation', 'B.3 SMMF Evaluation Details']


### Nothing! we need to find an rank_func

In [31]:
# We need to find another rank function
from llama_index.postprocessor.flag_embedding_reranker import FlagEmbeddingReranker
# rank_fn = FlagEmbeddingReranker(top_n=5, model='BAAI/bge-reranker-large')

### Part3:Adaptive ICL

In [32]:
# import base LLM
import os
import getpass

os.environ['CHATGLM4_API_KEY'] = getpass.getpass('CHATGLM4_API_KEY:')

CHATGLM4_API_KEY: ········


In [90]:
## laod LLM model
input_api_key = os.environ['CHATGLM4_API_KEY']
from dbgpt.model.proxy import ZhipuLLMClient
llm_client = ZhipuLLMClient(api_key=input_api_key)

In [91]:
# now we test this function
rewrite_instance = QueryRewrite(
    llm_client=llm_client,
    model_name="glm-4"
)

In [92]:
# We need a prompt context
context = retriever_chunks[-1].content

In [93]:
context

'B.3 SMMF Evaluation Details'

In [94]:
context = "SMMF is an inference framework"

In [95]:
type(context)

str

In [96]:
from dbgpt.rag.operators.rewrite import QueryRewriteOperator

In [97]:
reinforce = QueryRewriteOperator(
    llm_client=llm_client,
    model_name="glm-4",
)

In [98]:
query = "what is smmf"

In [99]:
input_dict = {"query": query, "context": context}

In [100]:
print(input_dict)

{'query': 'what is smmf', 'context': 'SMMF is an inference framework'}


In [101]:
await reinforce.map(query_context=input_dict)

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
[ModelOutput(text='[1214][prompt 参数非法。请检查文档。]', error_code=1, model_context=None, finish_reason=None, usage=None, metrics=None)]
rewrite queries: ['[1214][prompt 参数非法。请检查文档。]']


['[1214][prompt 参数非法。请检查文档。]']

In [102]:
from dbgpt.util.chat_util import run_async_tasks

In [103]:
REWRITE_PROMPT_TEMPLATE_EN = """
Based on the given context {context}, Generate {nums} search queries related to:
{original_query}, Provide following comma-separated format: 'queries: <queries>'":
    "original query:{original_query}\n"
    "queries:"
"""

In [104]:
origin_query = "what is smmf"
nums=1

In [105]:
prompt = REWRITE_PROMPT_TEMPLATE_EN.format(
    context=context, original_query=origin_query, nums=nums
)

In [106]:
prompt

'\nBased on the given context SMMF is an inference framework, Generate 1 search queries related to:\nwhat is smmf, Provide following comma-separated format: \'queries: <queries>\'":\n    "original query:what is smmf\n"\n    "queries:"\n'

In [107]:
from typing import List, Optional

from dbgpt.core import LLMClient, ModelMessage, ModelMessageRoleType, ModelRequest
from dbgpt.core.awel.flow import Parameter, ResourceCategory, register_resource
from dbgpt.util.i18n_utils import _

In [108]:
messages = [ModelMessage(role=ModelMessageRoleType.SYSTEM, content=prompt)]

In [109]:
print(messages)

[ModelMessage(role='system', content='\nBased on the given context SMMF is an inference framework, Generate 1 search queries related to:\nwhat is smmf, Provide following comma-separated format: \'queries: <queries>\'":\n    "original query:what is smmf\n"\n    "queries:"\n', round_index=0)]


In [114]:
request = ModelRequest(model="glm-4", messages=messages)
print(request)

ModelRequest(model='glm-4', messages=[ModelMessage(role='system', content='\nBased on the given context SMMF is an inference framework, Generate 1 search queries related to:\nwhat is smmf, Provide following comma-separated format: \'queries: <queries>\'":\n    "original query:what is smmf\n"\n    "queries:"\n', round_index=0)], temperature=None, max_new_tokens=None, stop=None, stop_token_ids=None, context_len=None, echo=False, span_id=None, context=ModelRequestContext(stream=False, cache_enable=False, user_name=None, sys_code=None, conv_uid=None, span_id=None, chat_mode=None, chat_param=None, extra={}, request_id=None))


In [115]:
tasks = [llm_client.generate(request)]

In [116]:
queries = await run_async_tasks(tasks=tasks)
print(queries)

[ModelOutput(text='[1214][prompt 参数非法。请检查文档。]', error_code=1, model_context=None, finish_reason=None, usage=None, metrics=None)]


## Write Your Own Chat Data With AWEL

In [2]:
# loading embedding model
from dbgpt.rag.embedding.embedding_factory import DefaultEmbeddingFactory
embeddings = DefaultEmbeddingFactory.call_embeddings_model_from_huggingface("BAAI/bge-base-en")

In [3]:
from dbgpt.datasource.rdbms.conn_sqlite import SQLiteTempConnector

db_conn = SQLiteTempConnector.create_temporary_db()
db_conn.create_temp_tables(
    {
        "user": {
            "columns": {
                "id": "INTEGER PRIMARY KEY",
                "name": "TEXT",
                "age": "INTEGER",
            },
            "data": [
                (1, "Tom", 10),
                (2, "Jerry", 16),
                (3, "Jack", 18),
                (4, "Alice", 20),
                (5, "Bob", 22),
            ],
        }
    }
)

In [4]:
# Store Database Schema To Vector Store

import asyncio
import shutil
from dbgpt.core.awel import DAG, InputOperator
from dbgpt.rag import ChunkParameters
from dbgpt.rag.operators import DBSchemaAssemblerOperator
from dbgpt.storage.vector_store.chroma_store import ChromaVectorConfig
from dbgpt.storage.vector_store.connector import VectorStoreConnector

# Delete old vector store directory(/tmp/awel_with_data_vector_store)
shutil.rmtree("/tmp/awel_with_data_vector_store", ignore_errors=True)

vector_connector = VectorStoreConnector.from_default(
    "Chroma",
    vector_store_config=ChromaVectorConfig(
        name="db_schema_vector_store",
        persist_path="/tmp/awel_with_data_vector_store",
    ),
    embedding_fn=embeddings
)

<class 'dbgpt.storage.vector_store.chroma_store.ChromaStore'>


In [5]:
with DAG("load_schema_dag") as load_schema_dag:
    input_task = InputOperator.dummy_input()
    # Load database schema to vector store
    assembler_task = DBSchemaAssemblerOperator(
        connector=db_conn,
        vector_store_connector=vector_connector,
        chunk_parameters=ChunkParameters(chunk_strategy="CHUNK_BY_SIZE")
    )
    input_task >> assembler_task

In [6]:
chunks = await assembler_task.call()

In [7]:
print(chunks)

[Chunk(content='user(id, name, age)', metadata={'source': 'database'}, chunk_id='9b41093b-8696-4885-9616-717b72f8c2bf', score=0.0, summary='', separator='\n')]


In [8]:
# Retrieve database schema from vector store
from dbgpt.core.awel import InputSource
from dbgpt.rag.operators import DBSchemaRetrieverOperator

with DAG("retrieve_schema_dag") as retrieve_schema_dag:
    input_task = InputOperator(input_source=InputSource.from_callable())
    retriever_task = DBSchemaRetrieverOperator(
        top_k=1,
        vector_store_connector=vector_connector,
    )
    input_task >> retriever_task
chunks = await retriever_task.call("Query the name and age pf users younger than 18 years old")

In [9]:
print("retrieved schema:\n", chunks)

retrieved schema:
 [Chunk(content='user(id, name, age)', metadata={'source': 'database'}, chunk_id='160319e4-b692-4a60-a6fe-18b2cdebef41', score=0.0, summary='', separator='\n')]


In [10]:
# prepare LLM
# import base LLM
import os
import getpass

os.environ['CHATGLM4_API_KEY'] = getpass.getpass('CHATGLM4_API_KEY:')

In [11]:
input_api_key = os.environ['CHATGLM4_API_KEY']
from dbgpt.model.proxy import ZhipuLLMClient
llm_client = ZhipuLLMClient(api_key=input_api_key)

In [12]:
# prepare some decisions
antv_charts = [
    {"response_line_chart": "used to display comparative trend analysis data"},
    {
        "response_pie_chart": "suitable for scenarios such as proportion and distribution statistics"
    },
    {
        "response_table": "suitable for display with many display columns or non-numeric columns"
    },
    # {"response_data_text":" the default display method, suitable for single-line or simple content display"},
    {
        "response_scatter_plot": "Suitable for exploring relationships between variables, detecting outliers, etc."
    },
    {
        "response_bubble_chart": "Suitable for relationships between multiple variables, highlighting outliers or special situations, etc."
    },
    {
        "response_donut_chart": "Suitable for hierarchical structure representation, category proportion display and highlighting key categories, etc."
    },
    {
        "response_area_chart": "Suitable for visualization of time series data, comparison of multiple groups of data, analysis of data change trends, etc."
    },
    {
        "response_heatmap": "Suitable for visual analysis of time series data, large-scale data sets, distribution of classified data, etc."
    },
]
display_type = "\n".join(
    f"{key}:{value}" for dict_item in antv_charts for key, value in dict_item.items()
)

In [13]:
# Generate SQL
import asyncio
import json

from dbgpt.core import(
    ChatPromptTemplate,
    HumanPromptTemplate,
    SystemPromptTemplate,
    SQLOutputParser
)

from dbgpt.core.awel import DAG, InputOperator, InputSource, MapOperator, JoinOperator
from dbgpt.core.operators import PromptBuilderOperator, RequestBuilderOperator
from dbgpt.rag.operators import DBSchemaRetrieverOperator
from dbgpt.model.operators import LLMOperator

In [14]:
system_prompt = """You are a database expert. Please answer the user's question based on the database selected by the user and some of the available table structure definitions of the database.
Database name:
    {db_name}
Table structure definition:
    {table_info}
    
Constraint:
1.Please understand the user's intention based on the user's question, and use the given table structure definition to create a grammatically correct {dialect} sql. If sql is not required, answer the user's question directly.. 
2.Always limit the query to a maximum of {top_k} results unless the user specifies in the question the specific number of rows of data he wishes to obtain.
3.You can only use the tables provided in the table structure information to generate sql. If you cannot generate sql based on the provided table structure, please say: "The table structure information provided is not enough to generate sql queries." It is prohibited to fabricate information at will.
4.Please be careful not to mistake the relationship between tables and columns when generating SQL.
5.Please check the correctness of the SQL and ensure that the query performance is optimized under correct conditions.
6.Please choose the best one from the display methods given below for data rendering, and put the type name into the name parameter value that returns the required format. If you cannot find the most suitable one, use 'Table' as the display method.
the available data display methods are as follows: {display_type}
 
User Question:
    {user_input}
Please think step by step and respond according to the following JSON format:
    {response}
Ensure the response is correct json and can be parsed by Python json.loads.
"""

In [15]:
RESPONSE_FORMAT_SIMPLE = {
    "thoughts": "thoughts summary to say to user",
    "sql": "SQL Query to run",
    "display_type": "Data display method",
}

In [16]:
prompt = ChatPromptTemplate(
    messages=[
        SystemPromptTemplate.from_template(
            system_prompt,
            response_format=json.dumps(
                RESPONSE_FORMAT_SIMPLE, ensure_ascii=False, indent=4
            ),
        ),
        HumanPromptTemplate.from_template("{user_input}"),
    ]
)

In [17]:
with DAG("chat_data_dag") as chat_data_dag:
    input_task = InputOperator(input_source=InputSource.from_callable())
    retriever_task = DBSchemaRetrieverOperator(
        top_k=1,
        vector_store_connector= vector_connector,
    )
    content_task = MapOperator(lambda cks: [c.content for c in cks])
    merge_task = JoinOperator(lambda table_info, ext_dict: {"table_info": table_info, **ext_dict})
    prompt_task = PromptBuilderOperator(prompt)
    req_build_task = RequestBuilderOperator(model="glm-4")
    llm_task = LLMOperator(llm_client=llm_client)
    sql_parse_task = SQLOutputParser()

    input_task >> MapOperator(lambda x: x["user_input"]) >> retriever_task >> content_task >> merge_task
    input_task >> merge_task
    merge_task >> prompt_task >> req_build_task >> llm_task >> sql_parse_task

In [18]:
result = await sql_parse_task.call(
    {
        "user_input":"Query the name and age of users younger than 18 years old",
        "db_name":"user_management",
        "dialect":"SQLite",
        "top_k":1,
        "display_type": display_type,
        "response": json.dumps(
            RESPONSE_FORMAT_SIMPLE, ensure_ascii=False, indent=4
        )
    }
)

print("Result:\n", result)

un_stream ai response: ```json
{
    "thoughts": "To retrieve the names and ages of users who are younger than 18 years old, we will use a SELECT statement with a WHERE clause to filter out the users based on their age. Since the user requested a specific number of results is not specified, we will limit the query to 1 result for readability.",
    "sql": "SELECT name, age FROM user WHERE age < 18 LIMIT 1",
    "display_type": "response_table"
}
```
Result:
 {'thoughts': 'To retrieve the names and ages of users who are younger than 18 years old, we will use a SELECT statement with a WHERE clause to filter out the users based on their age. Since the user requested a specific number of results is not specified, we will limit the query to 1 result for readability.', 'sql': 'SELECT name, age FROM user WHERE age < 18 LIMIT 1', 'display_type': 'response_table'}


In [19]:
print("Result:\n", result)

Result:
 {'thoughts': 'To retrieve the names and ages of users who are younger than 18 years old, we will use a SELECT statement with a WHERE clause to filter out the users based on their age. Since the user requested a specific number of results is not specified, we will limit the query to 1 result for readability.', 'sql': 'SELECT name, age FROM user WHERE age < 18 LIMIT 1', 'display_type': 'response_table'}


In [20]:
# Execute SQL
from dbgpt.datasource.operators.datasource_operator import DatasourceOperator

with DAG("chat_data_dag") as chat_data_dag:
    input_task = InputOperator(input_source=InputSource.from_callable())
    retriever_task = DBSchemaRetrieverOperator(
        top_k=1,
        vector_store_connector= vector_connector,
    )
    content_task = MapOperator(lambda cks: [c.content for c in cks])
    merge_task = JoinOperator(lambda table_info, ext_dict: {"table_info": table_info, **ext_dict})
    prompt_task = PromptBuilderOperator(prompt)
    req_build_task = RequestBuilderOperator(model="glm-4")
    llm_task = LLMOperator(llm_client=llm_client)
    sql_parse_task = SQLOutputParser()

    input_task >> MapOperator(lambda x: x["user_input"]) >> retriever_task >> content_task >> merge_task
    input_task >> merge_task
    merge_task >> prompt_task >> req_build_task >> llm_task >> sql_parse_task
    
    db_query_task = DatasourceOperator(connector=db_conn)
    sql_parse_task >> MapOperator(lambda x: x["sql"]) >> db_query_task
    
    db_result = await db_query_task.call(
        {
            "user_input": "Query the name and age of users younger than 18 years old",
            "db_name": "user_management",
            "dialect": "SQLite",
            "top_k": 1,
            "display_type": display_type,
            "response": json.dumps(RESPONSE_FORMAT_SIMPLE, ensure_ascii=False, indent=4)
        }
    )
    
    print("the result of the query is:")
    print(db_result)

un_stream ai response: ```json
{
    "thoughts": "To retrieve the names and ages of users who are younger than 18 years old, we will use a SELECT statement with a WHERE clause to filter the users based on their age.",
    "sql": "SELECT name, age FROM user WHERE age < 18 LIMIT 1;",
    "display_type": "response_table"
}
``` 

This JSON response provides a summary of the approach, the SQL query to execute, and the data display method which, in this case, is a table since we are returning non-numeric columns. The SQL query is limited to 1 result as per the constraint, unless the user specifies otherwise.
the result of the query is:
  name  age
0  Tom   10


In [22]:
# write your custom process logic after sql execution
import pandas  as pd

from dbgpt.core.awel import MapOperator, BranchOperator, JoinOperator, is_empty_data

class TwoSumOperator(MapOperator[pd.DataFrame, int]):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    async def map(self, df: pd.DataFrame) -> int:
        return await self.blocking_func_to_async(self._two_sum, df)

    def _two_sum(self, df: pd.DataFrame) -> int:
        return df['age'].sum()

def branch_even(x: int) -> bool:
    return x % 2 == 0

def branch_odd(x: int) -> bool:
    return not branch_even(x)

class DataDecisionOperator(BranchOperator[int, int]):
    def __init__(self, odd_task_name: str, even_task_name: str, **kwargs):
        super().__init__(**kwargs)
        self.odd_task_name = odd_task_name
        self.even_task_name = even_task_name

    async def branches(self):
        return {
            branch_even: self.even_task_name,
            branch_odd: self.odd_task_name
        }

class OddOperator(MapOperator[int, str]):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    async def map(self, x: int) -> str:
        print(f"{x} is odd")
        return f"{x} is odd"

class EvenOperator(MapOperator[int, str]):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    async def map(self, x: int) -> str:
        print(f"{x} is even")
        return f"{x} is even"

class MergeOperator(JoinOperator[str]):
    def __init__(self, **kwargs):
        super().__init__(combine_function=self.merge_func, **kwargs)

    async def merge_func(self, odd: str, even: str) -> str:
        return odd if not is_empty_data(odd) else even
        

In [23]:
with DAG("chat_data_dag") as chat_data_dag:
    input_task = InputOperator(input_source=InputSource.from_callable())
    retriever_task = DBSchemaRetrieverOperator(
        top_k=1,
        vector_store_connector= vector_connector,
    )
    content_task = MapOperator(lambda cks: [c.content for c in cks])
    merge_task = JoinOperator(lambda table_info, ext_dict: {"table_info": table_info, **ext_dict})
    prompt_task = PromptBuilderOperator(prompt)
    req_build_task = RequestBuilderOperator(model="glm-4")
    llm_task = LLMOperator(llm_client=llm_client)
    sql_parse_task = SQLOutputParser()

    input_task >> MapOperator(lambda x: x["user_input"]) >> retriever_task >> content_task >> merge_task
    input_task >> merge_task
    merge_task >> prompt_task >> req_build_task >> llm_task >> sql_parse_task
    
    db_query_task = DatasourceOperator(connector=db_conn)
    sql_parse_task >> MapOperator(lambda x: x["sql"]) >> db_query_task

    two_sum_task = TwoSumOperator()
    decision_task = DataDecisionOperator(odd_task_name="odd_task", even_task_name="even_task")
    odd_task = OddOperator(task_name="odd_task")
    even_task = EvenOperator(task_name="even_task")
    merge_task = MergeOperator()
    
    db_query_task >> two_sum_task >> decision_task
    decision_task >> odd_task >> merge_task
    decision_task >> even_task >> merge_task

final_result = await merge_task.call(
    {
        "user_input": "Query the name and age of users younger than 18 years old",
        "db_name": "user_management",
        "dialect": "SQLite",
        "top_k": 2,
        "display_type": display_type,
        "response": json.dumps(RESPONSE_FORMAT_SIMPLE, ensure_ascii=False, indent=4) 
    }
)

un_stream ai response: ```json
{
    "thoughts": "To retrieve the names and ages of users who are younger than 18 years old, we will use a simple SELECT statement with a WHERE clause to filter the age.",
    "sql": "SELECT name, age FROM user WHERE age < 18 LIMIT 2;",
    "display_type": "response_table"
}
```
26 is even


## dbgpt/core/interface

## API functions Test